# 🏙️ Spanish Cities Real Estate Market Analysis

**Purpose:** Comprehensive Spanish real estate market analysis across multiple cities  
**Dataset:** Spanish Real Estate Markets (All Available Cities)  
**Date:** December 2024  
**Environment:** Python 3.8+, pandas, numpy, plotly, matplotlib, seaborn

## 🎯 Analysis Objectives

- Automatically discover and process aall available cities with housing data
- Analyze sales and rental markets for each city independently
- Generate comprehensive market profiles for each urban area
- Perform cross-city comparative analysis and rankings
- Identify national market patterns and investment opportunities
- Export consolidated datasets ready for predictive modeling

## 📋 Metadata

- **Purpose:** Multi-city exploratory and comparative real estate analysis
- **Dataset version:** Raw Kaggle data - All available Spanish cities
- **Required environment:** Python 3.8+, pandas>=1.3.0, numpy>=1.21.0, matplotlib>=3.5.0, seaborn>=0.11.0
- **Date:** December 2024
- **Processing scope:** Automatically discovered from houses_*.csv files
- **Hardware requirements:** Minimum 4GB RAM for processing all cities


In [3]:
"""
Environment Setup and Configuration
"""
import sys
import warnings
from pathlib import Path

# Add project root to path for module imports
project_root = Path('..').resolve()
sys.path.insert(0, str(project_root))

# Data science libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Project modules
from src.data.exploratory_analysis import (
    load_data, clean_data, split_data, save_data, 
    create_mapping_dataframe, generate_basic_profile, 
    harmonize_datasets
)
from src.analysis.comparative_analysis import (
    safe_compare_markets, examine_data_structure
)

# Configuration
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', lambda x: '%.2f' % x)
sns.set_style("whitegrid")

# Try to import plotly with fallback
try:
    import plotly.express as px
    import plotly.graph_objects as go
    HAS_PLOTLY = True
    px.defaults.template = "plotly_white" 
    px.defaults.width = 1200
    px.defaults.height = 700
    print("✅ Plotly available for enhanced visualizations")
except ImportError:
    HAS_PLOTLY = False
    print("⚠️ Plotly not available. Using matplotlib for visualizations.")

# Define paths
PROVINCE = 'albacete'
DATA_PATH = Path('../data/raw/data-kaggle')
PROCESSED_PATH = Path('../data/processed/'+ PROVINCE)
FINAL_PATH = Path('../data/final/'+ PROVINCE)
REPORTS_PATH = Path('../reports/profiles')

# Create directories
for path in [PROCESSED_PATH, FINAL_PATH, REPORTS_PATH]:
    path.mkdir(parents=True, exist_ok=True)

print("\n✅ Environment setup complete!")
print(f"📁 Raw data path: {DATA_PATH}")
print(f"📁 Processed data path: {PROCESSED_PATH}")
print(f"📁 Final data path: {FINAL_PATH}")
print(f"💾 Reports path: {REPORTS_PATH}")


✅ Plotly available for enhanced visualizations

✅ Environment setup complete!
📁 Raw data path: ../data/raw/data-kaggle
📁 Processed data path: ../data/processed/albacete
📁 Final data path: ../data/final/albacete
💾 Reports path: ../reports/profiles


## 1. Data Loading and Initial Processing

First, we'll load and process the Álava housing data using our standardized pipeline.


In [4]:
# Load and process Álava housing data
province_file = DATA_PATH / f"houses_{PROVINCE}.csv"
print(f"🔄 Processing data from: {province_file}")

if not province_file.exists():
    print(f"❌ File not found: {province_file}")
    sys.exit(1)

# Load the raw data
raw_data = load_data(province_file)

if raw_data is None:
    print("❌ Failed to load data. Exiting...")
    sys.exit(1)

# Clean the data
cleaned_data, house_type_mapping = clean_data(raw_data)

if cleaned_data is None:
    print("❌ Failed to clean data. Exiting...")
    sys.exit(1)

# Split into rental and sales data
rental_data, sales_data = split_data(cleaned_data, house_type_mapping)

# Save processed datasets
if rental_data is not None:
    save_data(rental_data, PROCESSED_PATH / f"houses_{PROVINCE}_cleaned_rent.csv")

if sales_data is not None:
    save_data(sales_data, PROCESSED_PATH / f"houses_{PROVINCE}_cleaned_sale.csv")

# Save house type mapping
if house_type_mapping:
    mapping_df = create_mapping_dataframe(house_type_mapping, 'House_Type_Mapping')
    save_data(mapping_df, PROCESSED_PATH / 'houses_type_mapping.csv')

# Set variables for analysis
df_sales = sales_data
df_rental = rental_data

print(f"\n✅ Data processing complete!")
print(f"   Sales data: {df_sales.shape if df_sales is not None else 'None'}")
print(f"   Rental data: {df_rental.shape if df_rental is not None else 'None'}")


🔄 Processing data from: ../data/raw/data-kaggle/houses_albacete.csv
✅ Data loaded successfully from ../data/raw/data-kaggle/houses_albacete.csv
   Shape: (4469, 36)
   Memory usage: 10.10 MB
🧹 Cleaning and preprocessing data...
   Dropped columns: ['ground_size', 'kitchen', 'unfurnished', 'loc_street', 'ad_description']
   Applied one-hot encoding to: ['condition', 'heating', 'orientation']
✅ Data cleaned successfully. Final shape: (4469, 59)
⚠️ No rental types found in house_type mapping
💾 Data saved successfully to ../data/processed/albacete/houses_albacete_cleaned_sale.csv
   Shape: (4469, 59)
💾 Data saved successfully to ../data/processed/albacete/houses_type_mapping.csv
   Shape: (18, 2)

✅ Data processing complete!
   Sales data: (4469, 59)
   Rental data: None


## 2. Data Structure Examination

Let's examine the structure and content of our processed datasets.


In [5]:
# Examine data structures
if df_sales is not None:
    examine_data_structure(df_sales, "Sales Data")


=== SALES DATA STRUCTURE ===
Shape: (4469, 59)
Memory usage: 6.05 MB

Columns (59):
  - ad_last_update: object (4469 non-null, 0 null)
  - air_conditioner: object (4469 non-null, 0 null)
  - balcony: object (4469 non-null, 0 null)
  - bath_num: float64 (4469 non-null, 0 null)
  - built_in_wardrobe: object (4469 non-null, 0 null)
  - chimney: object (4469 non-null, 0 null)
  - construct_date: object (1083 non-null, 3386 null)
  - energetic_certif: object (2971 non-null, 1498 null)
  - floor: object (3388 non-null, 1081 null)
  - garage: int64 (4469 non-null, 0 null)
  - garden: object (4469 non-null, 0 null)
  - house_id: object (4469 non-null, 0 null)
  - house_type: int64 (4469 non-null, 0 null)
  - lift: object (2408 non-null, 2061 null)
  - loc_city: object (4469 non-null, 0 null)
  - loc_district: object (3808 non-null, 661 null)
  - loc_full: object (4469 non-null, 0 null)
  - loc_neigh: object (733 non-null, 3736 null)
  - loc_zone: object (4469 non-null, 0 null)
  - m2_real: ob

In [5]:
if df_rental is not None:
    examine_data_structure(df_rental, "Rental Data")

## 3. Profile Generation

Generate comprehensive profiles for both datasets to understand their characteristics.


In [6]:

# Generate profiles for both datasets
if df_sales is not None:
    sales_profile = generate_basic_profile(df_sales, f"{PROVINCE} Real Estate Sales")
    print("\n" + "="*60)

if df_rental is not None:
    rental_profile = generate_basic_profile(df_rental, f"{PROVINCE} Real Estate Rentals")


📈 Generating basic profile for: albacete Real Estate Sales
   DataFrame shape: (4469, 59)

   === PROFILE SUMMARY ===
   Rows: 4,469
   Columns: 59
   Duplicate rows: 14

   Missing values:
     - construct_date: 3386 (75.77%)
     - energetic_certif: 1498 (33.52%)
     - floor: 1081 (24.19%)
     - lift: 2061 (46.12%)
     - loc_district: 661 (14.79%)
     - loc_neigh: 3736 (83.6%)
     - m2_useful: 1935 (43.3%)



## 4. Data Harmonization and Export

Prepare harmonized datasets for comparative analysis and export to final directory.


In [7]:
# Harmonize and export datasets
if df_sales is not None or df_rental is not None:
    df_sales_final, df_rental_final = harmonize_datasets(
        df_sales, df_rental, FINAL_PATH, PROVINCE
    )
    
    print(f"\n📋 Sample of harmonized sales data:")
    print(df_sales_final.head(3).to_string())
    
    print(f"\n📋 Sample of harmonized rental data:")
    print(df_rental_final.head(3).to_string())
else:
    print("❌ Cannot harmonize datasets: Missing sales or rental data")


✅ Harmonized datasets saved to: ../data/final/albacete
   Sales: 4469 rows × 10 cols
   Rental: 0 rows × 10 cols

📋 Sample of harmonized sales data:
    price  bath_num  room_num  house_type  house_id m2_real m2_useful loc_city                   loc_zone construct_date
0  398000      3.00      7.00           0  83122970     414       400   Hellín  Campo de Hellín, Albacete            NaN
1   55000      1.00      3.00           1  81171345      85       NaN   Hellín  Campo de Hellín, Albacete            NaN
2   60000      2.00      3.00           1  81204694     125       NaN   Liétor  Campo de Hellín, Albacete            NaN

📋 Sample of harmonized rental data:
Empty DataFrame
Columns: [price, bath_num, room_num, house_type, house_id, m2_real, m2_useful, loc_city, loc_zone, construct_date]
Index: []


## 5. Comparative Market Analysis

Now let's perform a comprehensive comparison between the sales and rental markets.


In [8]:
# Perform comprehensive market comparison
if df_sales is not None and df_rental is not None:
    print("🔍 Starting comprehensive market comparison...")
    
    # Use the safe comparison function that handles all edge cases
    comparison_success = safe_compare_markets(
        df_sales, 
        df_rental,
        price_col='price',  # Let the function auto-detect if None
        surface_col=None    # Let the function auto-detect
    )
    
    if comparison_success:
        print("\n✅ Market comparison completed successfully!")
    else:
        print("\n⚠️ Market comparison had issues. Check data structure.")
        # Fallback: basic comparison
        print("\n=== BASIC COMPARISON FALLBACK ===")
        print(f"Sales data: {df_sales.shape[0]} records with {df_sales.shape[1]} features")
        print(f"Rental data: {df_rental.shape[0]} records with {df_rental.shape[1]} features")
else:
    print("❌ Cannot compare markets: Missing data for either sales or rentals")


❌ Cannot compare markets: Missing data for either sales or rentals


## 6. Summary and Conclusions

Key insights from the Álava real estate market analysis.


In [ ]:
# Summary of analysis
print("📊 ÁLAVA REAL ESTATE MARKET ANALYSIS SUMMARY")
print("="*50)

if df_sales is not None and df_rental is not None:
    print(f"✅ Analysis completed successfully")
    print(f"   📈 Sales market: {df_sales.shape[0]:,} properties analyzed")
    print(f"   🏠 Rental market: {df_rental.shape[0]:,} properties analyzed")
    print(f"   📁 Harmonized datasets exported to: {FINAL_PATH}")
    print(f"   🔍 Processed data saved to: {PROCESSED_PATH}")
    
    # Key metrics summary
    if 'price' in df_sales.columns and 'price' in df_rental.columns:
        avg_sale_price = df_sales['price'].mean()
        avg_rental_price = df_rental['price'].mean()
        print(f"\n💰 Key Price Metrics:")
        print(f"   Average sales price: €{avg_sale_price:,.2f}")
        print(f"   Average monthly rental: €{avg_rental_price:,.2f}")
        print(f"   Annual rental equivalent: €{avg_rental_price * 12:,.2f}")
        
        if avg_sale_price > 0:
            estimated_yield = (avg_rental_price * 12) / avg_sale_price * 100
            print(f"   Estimated gross rental yield: {estimated_yield:.2f}%")
    
    print(f"\n📋 Files generated:")
    print(f"   - {FINAL_PATH}/{PROVINCE}_sales_final.csv")
    print(f"   - {FINAL_PATH}/{PROVINCE}_rental_final.csv") 
    print(f"   - {FINAL_PATH}/{PROVINCE}_sales_describe.csv")
    print(f"   - {FINAL_PATH}/{PROVINCE}_rental_describe.csv")
    print(f"   - {PROCESSED_PATH}/houses_type_mapping.csv")
    
else:
    print("❌ Analysis incomplete due to data processing issues")

print(f"\n🎯 Analysis objectives achieved:")
print(f"   ✅ Datasets processed and cleaned")
print(f"   ✅ Market profiles generated") 
print(f"   ✅ Comparative analysis performed")
print(f"   ✅ Data harmonized and exported")
print(f"   ✅ Ready for predictive modeling")


📊 ÁLAVA REAL ESTATE MARKET ANALYSIS SUMMARY
✅ Analysis completed successfully
   📈 Sales market: 3,679 properties analyzed
   🏠 Rental market: 127 properties analyzed
   📁 Harmonized datasets exported to: ../data/final/alava
   🔍 Processed data saved to: ../data/processed

💰 Key Price Metrics:
   Average sales price: €244,807.69
   Average monthly rental: €1,015.21
   Annual rental equivalent: €12,182.55
   Estimated gross rental yield: 4.98%

📋 Files generated:
   - ../data/final/alava/alava_sales_final.csv
   - ../data/final/alava/alava_rental_final.csv
   - ../data/final/alava/alava_sales_describe.csv
   - ../data/final/alava/alava_rental_describe.csv
   - ../data/processed/houses_type_mapping.csv

🎯 Analysis objectives achieved:
   ✅ Datasets processed and cleaned
   ✅ Market profiles generated
   ✅ Comparative analysis performed
   ✅ Data harmonized and exported
   ✅ Ready for predictive modeling
